# REVIEWER NOTICE — CROSS-DATASET ABLATION (DOMAIN STRESS TEST)

---

## **EXPERIMENT_BOX: CROSS_DATASET**

This notebook is a **domain stress test** — NOT a performance estimation experiment.

---

## CRITICAL SCIENTIFIC SCOPE

| Aspect | Main Model Notebook | This Notebook |
|--------|---------------------|---------------|
| **Question** | Subject-wise generalization within cohort | Cross-dataset generalization (domain shift) |
| **Training** | Sleep-EDF Expanded (78 subjects) | Sleep-EDF Expanded (78 subjects) |
| **Test** | Held-out subjects (same dataset) | SHHS (completely different dataset) |
| **Purpose** | Performance estimation | Domain robustness stress test |

---

## THIS IS NOT A PERFORMANCE ESTIMATION EXPERIMENT

**What this notebook tests:**
- Whether features learned on Sleep-EDF generalize to a completely different dataset
- Robustness to domain shift (different recording equipment, protocols, populations)
- Lower bound of generalization performance

**What this notebook does NOT test:**
- Optimal performance on SHHS (would require training on SHHS)
- Fair comparison with SHHS-specific models
- Performance ceiling on SHHS

---

## EXPERIMENTAL PROTOCOL

1. **Train on Sleep-EDF Expanded** (all 78 subjects)
2. **Test on SHHS** (no training, no tuning)
3. **NO cross-validation on SHHS**
4. **NO hyperparameter tuning on SHHS**
5. **Repeat with 3-5 random seeds** for robust estimation
6. **Report mean ± std** across seeds

---

## EXPECTED RESULTS

Performance is expected to be **significantly lower** than in-domain evaluation because:
- SHHS uses different recording equipment
- SHHS has different patient populations (older, with sleep disorders)
- SHHS has different annotation protocols
- Feature distributions differ between datasets

**A performance drop is EXPECTED and INFORMATIVE** — it quantifies domain shift.

---

**Notebook generated**: `datetime.now().isoformat()` will be set at execution time.

# Cross-Dataset Ablation: Sleep-EDF Expanded → SHHS

## Section 0: Dataset Information

### Sleep-EDF Expanded (Training)
- **Source**: PhysioNet
- **Subjects**: 78 (sleep-cassette subset)
- **Age range**: 25-101 years
- **Recording**: Healthy subjects, home recordings
- **Channels**: EEG (Fpz-Cz), EOG, EMG

### SHHS (Test)
- **Source**: Sleep Heart Health Study (NSRR)
- **Subjects**: ~5,800 (large-scale clinical study)
- **Age range**: 40-90 years
- **Recording**: Patients with cardiovascular risk factors
- **Channels**: EEG (C3-A2, C4-A1), EOG, EMG, ECG

### Domain Shift Factors

| Factor | Sleep-EDF | SHHS |
|--------|-----------|------|
| Population | Healthy | Cardiovascular risk |
| Age bias | 25-101 (mixed) | 40-90 (older) |
| Recording setup | Home PSG | Portable PSG |
| EEG channels | Fpz-Cz | C3-A2/C4-A1 |
| Annotation | R&K/AASM hybrid | AASM |
| Sample rate | 100 Hz | 125 Hz |

## Section 1: Imports and Configuration

In [ ]:
# ===================================================================
# IMPORTS AND CONFIGURATION
# ===================================================================

import os
import sys
import json
import pickle
import numpy as np
import pandas as pd
from datetime import datetime
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Scientific computing
from scipy import stats

# Machine learning
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    cohen_kappa_score, confusion_matrix, classification_report,
    precision_recall_fscore_support, matthews_corrcoef
)

# XGBoost
import xgboost as xgb

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ===================================================================
# EXPERIMENT CONFIGURATION
# ===================================================================

# Experiment identification (CRITICAL for checkpoint traceability)
EXPERIMENT_BOX = "CROSS_DATASET"  # Distinguishes from MAIN and CROSS_COHORT
DATASET_NAME = "Sleep-EDF Expanded → SHHS"  # Train on Sleep-EDF, test on SHHS

# Multiple random seeds for robust estimation
RANDOM_SEEDS = [42, 123, 456, 789, 1024]  # 5 seeds
N_SEEDS = len(RANDOM_SEEDS)

# Paths
PROJECT_ROOT = Path.cwd()
CHECKPOINT_DIR = PROJECT_ROOT / 'cache' / 'checkpoints'
RESULTS_DIR = PROJECT_ROOT / 'results'
OUTPUT_DIR = RESULTS_DIR / 'cross_dataset_shhs'

# Data directories
SLEEP_EDF_DIR = PROJECT_ROOT / 'sleep-edfx'
SHHS_DIR = PROJECT_ROOT / 'shhs'  # Expected location for SHHS data

# Feature cache
FEATURES_CACHE = PROJECT_ROOT / 'cache' / 'features'
SHHS_FEATURES_CACHE = PROJECT_ROOT / 'cache' / 'features_shhs'

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Sleep stage mapping (AASM)
STAGE_MAPPING = {
    0: 'Wake',
    1: 'N1',
    2: 'N2',
    3: 'N3',
    4: 'REM'
}

print("Cross-Dataset Ablation Environment")
print("="*70)
print(f"EXPERIMENT_BOX: {EXPERIMENT_BOX}")
print(f"DATASET_NAME:   {DATASET_NAME}")
print(f"RANDOM_SEEDS:   {RANDOM_SEEDS}")
print("-"*70)
print(f"Project root:   {PROJECT_ROOT}")
print(f"Output dir:     {OUTPUT_DIR}")
print(f"Sleep-EDF dir:  {SLEEP_EDF_DIR}")
print(f"SHHS dir:       {SHHS_DIR}")
print("="*70)

# Check data availability
SLEEP_EDF_AVAILABLE = SLEEP_EDF_DIR.exists()
SHHS_AVAILABLE = SHHS_DIR.exists()

print(f"\n📊 Data Availability:")
print(f"   Sleep-EDF Expanded: {'✓' if SLEEP_EDF_AVAILABLE else '✗ NOT FOUND'}")
print(f"   SHHS:               {'✓' if SHHS_AVAILABLE else '✗ NOT FOUND'}")

if not SHHS_AVAILABLE:
    print("\n" + "!"*70)
    print("WARNING: SHHS data not found!")
    print("To run this notebook, please:")
    print("1. Download SHHS data from NSRR (https://sleepdata.org/datasets/shhs)")
    print("2. Place it in the 'shhs/' directory")
    print("3. Preprocess features using the same pipeline as Sleep-EDF")
    print("!"*70)

## Section 2: Evaluation Metrics

In [ ]:
# ===================================================================
# EVALUATION METRICS
# ===================================================================

def compute_all_metrics(y_true, y_pred):
    """
    Compute comprehensive evaluation metrics for sleep staging.
    
    Parameters:
    -----------
    y_true : array-like
        Ground truth labels
    y_pred : array-like
        Predicted labels
        
    Returns:
    --------
    dict : Dictionary containing all metrics
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Overall metrics
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro'),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted'),
        'kappa': cohen_kappa_score(y_true, y_pred),
        'mcc': matthews_corrcoef(y_true, y_pred)
    }
    
    # Per-class metrics
    classes = np.unique(np.concatenate([y_true, y_pred]))
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=classes, zero_division=0
    )
    
    classes_list = np.asarray(classes).tolist()
    precision_list = np.asarray(precision).tolist()
    recall_list = np.asarray(recall).tolist()
    f1_list = np.asarray(f1).tolist()
    support_list = np.asarray(support).tolist()
    
    metrics['per_class_precision'] = dict(zip(classes_list, precision_list))
    metrics['per_class_recall'] = dict(zip(classes_list, recall_list))
    metrics['per_class_f1'] = dict(zip(classes_list, f1_list))
    metrics['per_class_support'] = dict(zip(classes_list, support_list))
    metrics['confusion_matrix'] = confusion_matrix(y_true, y_pred, labels=classes)
    
    return metrics


def aggregate_seed_results(all_results):
    """
    Aggregate results across multiple random seeds.
    
    Parameters:
    -----------
    all_results : list of dict
        List of metric dictionaries from each seed
        
    Returns:
    --------
    dict : Aggregated metrics with mean and std
    """
    # Metrics to aggregate
    metric_keys = ['accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1', 'kappa', 'mcc']
    
    aggregated = {}
    
    for key in metric_keys:
        values = [r[key] for r in all_results if key in r]
        if values:
            aggregated[f'{key}_mean'] = np.mean(values)
            aggregated[f'{key}_std'] = np.std(values)
            aggregated[f'{key}_values'] = values
    
    aggregated['n_seeds'] = len(all_results)
    
    return aggregated


print("✓ Evaluation metrics defined")
print("✓ Multi-seed aggregation function defined")

## Section 3: Load Training Data (Sleep-EDF Expanded)

In [ ]:
# ===================================================================
# LOAD TRAINING DATA (SLEEP-EDF EXPANDED)
# ===================================================================
"""
Training data comes from Sleep-EDF Expanded (78 subjects).
This is the SAME data used in the main model notebook.
We use cached features to ensure consistency.
"""

print("="*70)
print("LOADING TRAINING DATA (SLEEP-EDF EXPANDED)")
print("="*70)

def load_sleep_edf_features():
    """Load preprocessed Sleep-EDF Expanded features."""
    X_path = FEATURES_CACHE / 'X_df.pkl'
    y_path = FEATURES_CACHE / 'y.pkl'
    subjects_path = FEATURES_CACHE / 'subjects.pkl'
    feature_names_path = FEATURES_CACHE / 'feature_names.pkl'
    
    if not X_path.exists():
        raise FileNotFoundError(
            f"Sleep-EDF features not found: {FEATURES_CACHE}\n"
            "Please run the main notebook (01_main_model.ipynb) first to generate features."
        )
    
    X_df = pd.read_pickle(X_path)
    y = np.array(pd.read_pickle(y_path))
    subjects_raw = pd.read_pickle(subjects_path)
    
    # Normalize to 6-character format
    subjects = np.array([str(s)[:6] for s in subjects_raw])
    
    feature_names = pd.read_pickle(feature_names_path) if feature_names_path.exists() else X_df.columns.tolist()
    
    return X_df, y, subjects, feature_names


try:
    X_train_df, y_train, train_subjects, feature_names = load_sleep_edf_features()
    
    TRAIN_SUBJECT_IDS = sorted(np.unique(train_subjects).tolist())
    
    print(f"\n✓ Training data loaded (Sleep-EDF Expanded)")
    print(f"  Subjects: {len(TRAIN_SUBJECT_IDS)}")
    print(f"  Epochs:   {len(y_train)}")
    print(f"  Features: {len(feature_names)}")
    print(f"  Class distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}")
    
    TRAIN_DATA_LOADED = True
    
except FileNotFoundError as e:
    print(f"\n❌ {e}")
    TRAIN_DATA_LOADED = False
    X_train_df, y_train, train_subjects, feature_names = None, None, None, []

## Section 4: Load Test Data (SHHS)

In [ ]:
# ===================================================================
# LOAD TEST DATA (SHHS)
# ===================================================================
"""
Test data comes from SHHS (Sleep Heart Health Study).
This is a COMPLETELY DIFFERENT dataset with different:
- Population (cardiovascular patients vs healthy)
- Recording equipment
- Annotation protocols
- EEG channel montage

NO training or tuning is performed on SHHS — purely evaluation.
"""

print("="*70)
print("LOADING TEST DATA (SHHS)")
print("="*70)

def load_shhs_features():
    """
    Load preprocessed SHHS features.
    
    Features must be extracted using the SAME pipeline as Sleep-EDF
    to ensure fair comparison.
    """
    X_path = SHHS_FEATURES_CACHE / 'X_df.pkl'
    y_path = SHHS_FEATURES_CACHE / 'y.pkl'
    subjects_path = SHHS_FEATURES_CACHE / 'subjects.pkl'
    
    if not X_path.exists():
        raise FileNotFoundError(
            f"SHHS features not found: {SHHS_FEATURES_CACHE}\n"
            "Please preprocess SHHS data using the same feature extraction pipeline as Sleep-EDF.\n"
            "Required files:\n"
            f"  - {X_path}\n"
            f"  - {y_path}\n"
            f"  - {subjects_path}"
        )
    
    X_df = pd.read_pickle(X_path)
    y = np.array(pd.read_pickle(y_path))
    subjects_raw = pd.read_pickle(subjects_path)
    subjects = np.array([str(s) for s in subjects_raw])
    
    return X_df, y, subjects


try:
    X_test_df, y_test, test_subjects = load_shhs_features()
    
    TEST_SUBJECT_IDS = sorted(np.unique(test_subjects).tolist())
    
    print(f"\n✓ Test data loaded (SHHS)")
    print(f"  Subjects: {len(TEST_SUBJECT_IDS)}")
    print(f"  Epochs:   {len(y_test)}")
    print(f"  Class distribution: {dict(zip(*np.unique(y_test, return_counts=True)))}")
    
    # Feature alignment check
    if TRAIN_DATA_LOADED:
        common_features = set(feature_names) & set(X_test_df.columns)
        missing_in_shhs = set(feature_names) - set(X_test_df.columns)
        
        print(f"\n📊 Feature Alignment:")
        print(f"   Training features: {len(feature_names)}")
        print(f"   SHHS features:     {len(X_test_df.columns)}")
        print(f"   Common features:   {len(common_features)}")
        
        if missing_in_shhs:
            print(f"   ⚠️ Missing in SHHS: {len(missing_in_shhs)}")
    
    TEST_DATA_LOADED = True
    
except FileNotFoundError as e:
    print(f"\n⚠️ SHHS data not available")
    print(f"   {e}")
    print("\n" + "─"*70)
    print("📝 TO PREPARE SHHS DATA:")
    print("─"*70)
    print("1. Download SHHS-1 from NSRR (https://sleepdata.org/datasets/shhs)")
    print("2. Extract EEG epochs (30-second windows)")
    print("3. Use the SAME feature extraction pipeline as Sleep-EDF")
    print("4. Save to cache/features_shhs/:")
    print("   - X_df.pkl (feature matrix)")
    print("   - y.pkl (sleep stage labels)")
    print("   - subjects.pkl (subject IDs)")
    print("─"*70)
    
    TEST_DATA_LOADED = False
    X_test_df, y_test, test_subjects = None, None, None
    TEST_SUBJECT_IDS = []

## Section 5: Model Training (Multi-Seed)

In [ ]:
# ===================================================================
# MULTI-SEED MODEL TRAINING AND EVALUATION
# ===================================================================
"""
EXPERIMENTAL PROTOCOL:
1. Train on ALL Sleep-EDF Expanded subjects (78)
2. Test on ALL SHHS subjects
3. NO cross-validation on SHHS
4. NO hyperparameter tuning on SHHS
5. Repeat with 5 random seeds
6. Report mean ± std
"""

print("="*70)
print("MULTI-SEED TRAINING AND CROSS-DATASET EVALUATION")
print("="*70)
print(f"⚠️ Training on:   Sleep-EDF Expanded ({len(TRAIN_SUBJECT_IDS) if TRAIN_DATA_LOADED else 0} subjects)")
print(f"⚠️ Testing on:    SHHS ({len(TEST_SUBJECT_IDS) if TEST_DATA_LOADED else 0} subjects)")
print(f"⚠️ Random seeds:  {RANDOM_SEEDS}")
print(f"⚠️ NO tuning on SHHS — pure domain stress test")
print("="*70)


class WeightedEnsemble:
    """Weighted probability ensemble of XGBoost and LinearSVC."""
    
    def __init__(self, xgb_model, svc_model, xgb_weight=0.7, svc_weight=0.3):
        self.xgb_model = xgb_model
        self.svc_model = svc_model
        self.xgb_weight = xgb_weight
        self.svc_weight = svc_weight
    
    def predict_proba(self, X):
        xgb_proba = self.xgb_model.predict_proba(X)
        svc_proba = self.svc_model.predict_proba(X)
        return self.xgb_weight * xgb_proba + self.svc_weight * svc_proba
    
    def predict(self, X):
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1)


def train_and_evaluate_single_seed(seed, X_train, y_train, X_test, y_test, feature_names):
    """
    Train model with a specific seed and evaluate on test set.
    
    Parameters:
    -----------
    seed : int
        Random seed for reproducibility
    X_train, y_train : training data
    X_test, y_test : test data
    feature_names : list of feature names
    
    Returns:
    --------
    dict : Results including metrics, model, and metadata
    """
    print(f"\n{'─'*50}")
    print(f"SEED: {seed}")
    print(f"{'─'*50}")
    
    np.random.seed(seed)
    
    # Feature alignment
    common_features = [f for f in feature_names if f in X_test.columns]
    X_train_aligned = X_train[common_features].values
    X_test_aligned = X_test[common_features].values
    
    # Scale features (fit on training only)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_aligned)
    X_test_scaled = scaler.transform(X_test_aligned)
    
    print(f"  Features aligned: {len(common_features)}")
    
    # Train XGBoost
    print(f"  Training XGBoost...")
    xgb_model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        min_child_weight=1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        objective='multi:softprob',
        num_class=5,
        use_label_encoder=False,
        eval_metric='mlogloss',
        n_jobs=-1
    )
    xgb_model.fit(X_train_scaled, y_train)
    
    # Train Calibrated LinearSVC
    print(f"  Training LinearSVC...")
    base_svc = LinearSVC(
        C=1.0,
        max_iter=5000,
        random_state=seed,
        dual='auto'
    )
    svc_model = CalibratedClassifierCV(
        estimator=base_svc,
        method='sigmoid',
        cv=5
    )
    svc_model.fit(X_train_scaled, y_train)
    
    # Create ensemble
    ensemble = WeightedEnsemble(xgb_model, svc_model, xgb_weight=0.7, svc_weight=0.3)
    
    # Evaluate on test set (SHHS)
    print(f"  Evaluating on SHHS...")
    y_pred = ensemble.predict(X_test_scaled)
    
    # Compute metrics
    metrics = compute_all_metrics(y_test, y_pred)
    
    print(f"  Results:")
    print(f"    Accuracy:      {metrics['accuracy']:.4f}")
    print(f"    Balanced Acc:  {metrics['balanced_accuracy']:.4f}")
    print(f"    Macro F1:      {metrics['macro_f1']:.4f}")
    print(f"    Cohen's Kappa: {metrics['kappa']:.4f}")
    
    return {
        'seed': seed,
        'metrics': metrics,
        'model': ensemble,
        'scaler': scaler,
        'feature_names': common_features,
        'y_pred': y_pred,
    }


# Run multi-seed training and evaluation
all_seed_results = []

if TRAIN_DATA_LOADED and TEST_DATA_LOADED:
    
    for seed in RANDOM_SEEDS:
        result = train_and_evaluate_single_seed(
            seed=seed,
            X_train=X_train_df,
            y_train=y_train,
            X_test=X_test_df,
            y_test=y_test,
            feature_names=feature_names
        )
        all_seed_results.append(result)
    
    TRAINING_COMPLETE = True
    print(f"\n✓ Multi-seed training complete ({len(all_seed_results)} seeds)")
    
else:
    print("\n❌ Cannot run training: data not available")
    if not TRAIN_DATA_LOADED:
        print("   - Sleep-EDF features missing")
    if not TEST_DATA_LOADED:
        print("   - SHHS features missing")
    TRAINING_COMPLETE = False

## Section 6: Aggregate Results (Mean ± Std)

In [ ]:
# ===================================================================
# AGGREGATE RESULTS ACROSS SEEDS
# ===================================================================

print("="*70)
print("AGGREGATED RESULTS (MEAN ± STD ACROSS SEEDS)")
print("="*70)

if TRAINING_COMPLETE and len(all_seed_results) > 0:
    
    # Extract metrics from all seeds
    metrics_list = [r['metrics'] for r in all_seed_results]
    aggregated = aggregate_seed_results(metrics_list)
    
    # Primary metrics table
    print("\n📊 PRIMARY METRICS (Required for reporting)")
    print("─"*60)
    print(f"{'Metric':<25} {'Mean':>12} {'Std':>12} {'Range':>20}")
    print("─"*60)
    
    for metric in ['macro_f1', 'balanced_accuracy', 'kappa']:
        mean_val = aggregated[f'{metric}_mean']
        std_val = aggregated[f'{metric}_std']
        values = aggregated[f'{metric}_values']
        range_str = f"[{min(values):.4f}, {max(values):.4f}]"
        
        # Format metric name
        metric_name = metric.replace('_', ' ').title()
        
        print(f"{metric_name:<25} {mean_val:>12.4f} {std_val:>12.4f} {range_str:>20}")
    
    print("─"*60)
    
    # Additional metrics
    print("\n📊 ADDITIONAL METRICS")
    print("─"*60)
    
    for metric in ['accuracy', 'weighted_f1', 'mcc']:
        mean_val = aggregated[f'{metric}_mean']
        std_val = aggregated[f'{metric}_std']
        metric_name = metric.replace('_', ' ').title()
        print(f"{metric_name:<25} {mean_val:>12.4f} ± {std_val:.4f}")
    
    print("─"*60)
    print(f"\n📈 Number of seeds: {aggregated['n_seeds']}")
    print(f"📈 Seeds used: {RANDOM_SEEDS}")
    
    # Per-seed breakdown
    print("\n" + "="*70)
    print("PER-SEED BREAKDOWN")
    print("="*70)
    print(f"{'Seed':>8} {'Accuracy':>12} {'Macro F1':>12} {'Balanced Acc':>14} {'Kappa':>12}")
    print("─"*60)
    
    for result in all_seed_results:
        m = result['metrics']
        print(f"{result['seed']:>8} {m['accuracy']:>12.4f} {m['macro_f1']:>12.4f} {m['balanced_accuracy']:>14.4f} {m['kappa']:>12.4f}")
    
    print("─"*60)
    
    AGGREGATION_COMPLETE = True
    
else:
    print("\n❌ No results to aggregate")
    aggregated = None
    AGGREGATION_COMPLETE = False

## Section 7: Save Checkpoints and Results

In [ ]:
# ===================================================================
# SAVE CHECKPOINTS WITH REQUIRED FORMAT
# ===================================================================
"""
Save checkpoints with the required format:
{
  "experiment_box": "CROSS_DATASET",
  "dataset_name": "Sleep-EDF Expanded → SHHS",
  "random_seed": int,
  "train_subjects": list[str],
  "test_subjects": list[str],
  "metrics": dict
}
"""

print("="*70)
print("SAVING CHECKPOINTS AND RESULTS")
print("="*70)

if TRAINING_COMPLETE and AGGREGATION_COMPLETE:
    
    # Create checkpoint directory for this experiment
    checkpoint_subdir = CHECKPOINT_DIR / 'cross_dataset_shhs'
    checkpoint_subdir.mkdir(parents=True, exist_ok=True)
    
    # Save individual seed checkpoints
    for result in all_seed_results:
        seed = result['seed']
        
        checkpoint = {
            # === REQUIRED FIELDS ===
            'experiment_box': EXPERIMENT_BOX,
            'dataset_name': DATASET_NAME,
            'random_seed': seed,
            'train_subjects': TRAIN_SUBJECT_IDS,
            'test_subjects': TEST_SUBJECT_IDS,
            'metrics': {
                'accuracy': float(result['metrics']['accuracy']),
                'balanced_accuracy': float(result['metrics']['balanced_accuracy']),
                'macro_f1': float(result['metrics']['macro_f1']),
                'weighted_f1': float(result['metrics']['weighted_f1']),
                'kappa': float(result['metrics']['kappa']),
                'mcc': float(result['metrics']['mcc']),
                'per_class_recall': {int(k): float(v) for k, v in result['metrics']['per_class_recall'].items()},
                'per_class_f1': {int(k): float(v) for k, v in result['metrics']['per_class_f1'].items()},
                'confusion_matrix': result['metrics']['confusion_matrix'].tolist(),
            },
            
            # === ADDITIONAL FIELDS ===
            'model': result['model'],
            'scaler': result['scaler'],
            'feature_names': result['feature_names'],
            'timestamp': datetime.now().isoformat(),
            'training_config': {
                'model_type': 'cross_dataset_shhs',
                'ensemble_weights': {'xgb': 0.7, 'svc': 0.3},
                'n_train_subjects': len(TRAIN_SUBJECT_IDS),
                'n_test_subjects': len(TEST_SUBJECT_IDS),
            },
        }
        
        # Validate required fields
        REQUIRED_FIELDS = ['experiment_box', 'dataset_name', 'random_seed', 'train_subjects', 'test_subjects', 'metrics']
        missing = [f for f in REQUIRED_FIELDS if f not in checkpoint]
        if missing:
            raise ValueError(f"Checkpoint missing required fields: {missing}")
        
        # Save checkpoint
        checkpoint_path = checkpoint_subdir / f'seed_{seed}_checkpoint.pkl'
        with open(checkpoint_path, 'wb') as f:
            pickle.dump(checkpoint, f)
        
        print(f"✓ Checkpoint saved: {checkpoint_path.name}")
    
    print(f"\n✓ All {len(all_seed_results)} seed checkpoints saved")
    CHECKPOINTS_SAVED = True
    
else:
    print("\n❌ Cannot save checkpoints: training not complete")
    CHECKPOINTS_SAVED = False

## Section 8: Output Results to results/cross_dataset_shhs/

In [ ]:
# ===================================================================
# OUTPUT RESULTS TO results/cross_dataset_shhs/
# ===================================================================

print("="*70)
print("SAVING RESULTS TO results/cross_dataset_shhs/")
print("="*70)

if TRAINING_COMPLETE and AGGREGATION_COMPLETE:
    
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # 1. AGGREGATED METRICS CSV
    agg_csv_path = OUTPUT_DIR / 'cross_dataset_metrics_aggregated.csv'
    agg_df = pd.DataFrame([{
        'experiment_box': EXPERIMENT_BOX,
        'dataset_name': DATASET_NAME,
        'n_seeds': aggregated['n_seeds'],
        'macro_f1_mean': aggregated['macro_f1_mean'],
        'macro_f1_std': aggregated['macro_f1_std'],
        'balanced_accuracy_mean': aggregated['balanced_accuracy_mean'],
        'balanced_accuracy_std': aggregated['balanced_accuracy_std'],
        'kappa_mean': aggregated['kappa_mean'],
        'kappa_std': aggregated['kappa_std'],
        'accuracy_mean': aggregated['accuracy_mean'],
        'accuracy_std': aggregated['accuracy_std'],
    }])
    agg_df.to_csv(agg_csv_path, index=False)
    print(f"✓ Aggregated metrics: {agg_csv_path}")
    
    # 2. PER-SEED METRICS CSV
    seed_csv_path = OUTPUT_DIR / 'cross_dataset_metrics_per_seed.csv'
    seed_data = []
    for result in all_seed_results:
        m = result['metrics']
        seed_data.append({
            'seed': result['seed'],
            'accuracy': m['accuracy'],
            'balanced_accuracy': m['balanced_accuracy'],
            'macro_f1': m['macro_f1'],
            'weighted_f1': m['weighted_f1'],
            'kappa': m['kappa'],
            'mcc': m['mcc'],
        })
    seed_df = pd.DataFrame(seed_data)
    seed_df.to_csv(seed_csv_path, index=False)
    print(f"✓ Per-seed metrics: {seed_csv_path}")
    
    # 3. FULL METADATA JSON
    metadata_path = OUTPUT_DIR / 'cross_dataset_metadata.json'
    metadata = {
        'experiment_box': EXPERIMENT_BOX,
        'dataset_name': DATASET_NAME,
        'timestamp': datetime.now().isoformat(),
        'protocol': {
            'type': 'domain_stress_test',
            'training_dataset': 'Sleep-EDF Expanded',
            'test_dataset': 'SHHS',
            'cross_validation_on_test': False,
            'hyperparameter_tuning_on_test': False,
            'n_seeds': len(RANDOM_SEEDS),
            'seeds': RANDOM_SEEDS,
        },
        'train_subjects': TRAIN_SUBJECT_IDS,
        'test_subjects': TEST_SUBJECT_IDS,
        'aggregated_metrics': {
            'macro_f1': {
                'mean': aggregated['macro_f1_mean'],
                'std': aggregated['macro_f1_std'],
            },
            'balanced_accuracy': {
                'mean': aggregated['balanced_accuracy_mean'],
                'std': aggregated['balanced_accuracy_std'],
            },
            'kappa': {
                'mean': aggregated['kappa_mean'],
                'std': aggregated['kappa_std'],
            },
        },
        'clarifications': {
            'is_domain_stress_test': True,
            'is_performance_estimation': False,
            'expected_performance_drop': True,
            'reason': 'Cross-dataset evaluation tests domain shift, not optimal performance',
        },
    }
    
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"✓ Metadata JSON: {metadata_path}")
    
    # 4. CONFUSION MATRIX (from first seed)
    if len(all_seed_results) > 0:
        cm = all_seed_results[0]['metrics']['confusion_matrix']
        cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Raw counts
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=list(STAGE_MAPPING.values()),
                    yticklabels=list(STAGE_MAPPING.values()),
                    ax=axes[0])
        axes[0].set_xlabel('Predicted')
        axes[0].set_ylabel('True')
        axes[0].set_title(f'Confusion Matrix (Seed {all_seed_results[0]["seed"]})')
        
        # Normalized
        sns.heatmap(cm_normalized, annot=True, fmt='.3f', cmap='Blues',
                    xticklabels=list(STAGE_MAPPING.values()),
                    yticklabels=list(STAGE_MAPPING.values()),
                    ax=axes[1])
        axes[1].set_xlabel('Predicted')
        axes[1].set_ylabel('True')
        axes[1].set_title('Confusion Matrix (Normalized)')
        
        plt.suptitle(f'Cross-Dataset Evaluation: {DATASET_NAME}', fontsize=12, fontweight='bold')
        plt.tight_layout()
        
        fig_path = OUTPUT_DIR / 'confusion_matrix.png'
        plt.savefig(fig_path, dpi=300, bbox_inches='tight')
        print(f"✓ Confusion matrix: {fig_path}")
        plt.show()
    
    # 5. SEED COMPARISON PLOT
    fig, ax = plt.subplots(figsize=(10, 6))
    
    seeds = [r['seed'] for r in all_seed_results]
    macro_f1s = [r['metrics']['macro_f1'] for r in all_seed_results]
    kappas = [r['metrics']['kappa'] for r in all_seed_results]
    balanced_accs = [r['metrics']['balanced_accuracy'] for r in all_seed_results]
    
    x = np.arange(len(seeds))
    width = 0.25
    
    bars1 = ax.bar(x - width, macro_f1s, width, label='Macro F1', color='#2196F3')
    bars2 = ax.bar(x, balanced_accs, width, label='Balanced Acc', color='#4CAF50')
    bars3 = ax.bar(x + width, kappas, width, label='Cohen Kappa', color='#FF9800')
    
    ax.set_xlabel('Random Seed')
    ax.set_ylabel('Score')
    ax.set_title(f'Cross-Dataset Performance Across Seeds\n{DATASET_NAME}')
    ax.set_xticks(x)
    ax.set_xticklabels(seeds)
    ax.legend()
    ax.set_ylim(0, 1)
    ax.axhline(y=aggregated['macro_f1_mean'], color='#2196F3', linestyle='--', alpha=0.5)
    ax.axhline(y=aggregated['kappa_mean'], color='#FF9800', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    
    fig_path = OUTPUT_DIR / 'seed_comparison.png'
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    print(f"✓ Seed comparison: {fig_path}")
    plt.show()
    
    print(f"\n📁 All results saved to: {OUTPUT_DIR}")
    RESULTS_SAVED = True
    
else:
    print("\n❌ Cannot save results: training not complete")
    RESULTS_SAVED = False

In [ ]:
# ===================================================================
# STANDARDIZED PER-CLASS RECALL BAR PLOT WITH ERROR BARS
# ===================================================================
"""
Standardized visualization following project conventions:
- Fixed class order: ['W', 'N1', 'N2', 'N3', 'REM']
- Blues colormap for consistency
- Y-axis [0, 1] for recall metrics
- Error bars show std across seeds (Domain Stress Test)
- 300 DPI output
"""

if TRAINING_COMPLETE and len(all_seed_results) > 0:
    
    print("="*80)
    print("STANDARDIZED PER-CLASS RECALL VISUALIZATION (DOMAIN STRESS TEST)")
    print("="*80)
    
    # Fixed class order for consistency across all notebooks
    CLASS_ORDER = ['W', 'N1', 'N2', 'N3', 'REM']
    
    # Compute mean ± std per class across all seeds
    per_class_recalls = {i: [] for i in range(5)}
    
    for result in all_seed_results:
        for class_idx in range(5):
            recall_val = result['metrics']['per_class_recall'].get(class_idx, 0)
            per_class_recalls[class_idx].append(recall_val)
    
    mean_recalls = [np.mean(per_class_recalls[i]) for i in range(5)]
    std_recalls = [np.std(per_class_recalls[i]) for i in range(5)]
    
    # Create figure
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x = np.arange(len(CLASS_ORDER))
    colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(CLASS_ORDER)))
    
    # Bar plot with error bars
    bars = ax.bar(x, mean_recalls, width=0.6, color=colors, edgecolor='navy', linewidth=1,
                  yerr=std_recalls, capsize=5, error_kw={'linewidth': 1.5, 'color': 'black'})
    
    # Add value labels on top of bars
    for i, (bar, mean_val, std_val) in enumerate(zip(bars, mean_recalls, std_recalls)):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std_val + 0.03,
                f'{mean_val:.3f}±{std_val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Formatting
    ax.set_xlabel('Sleep Stage', fontsize=12, fontweight='bold')
    ax.set_ylabel('Recall', fontsize=12, fontweight='bold')
    ax.set_title(f'Per-Class Recall — Domain Stress Test (Cross-Dataset)\n{DATASET_NAME} | {len(all_seed_results)} Seeds', 
                 fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_ORDER, fontsize=11)
    ax.set_ylim(0, 1.0)
    ax.set_yticks(np.arange(0, 1.1, 0.1))
    
    # Add horizontal reference lines
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, linewidth=1, label='Chance (0.5)')
    overall_mean = np.mean(mean_recalls)
    ax.axhline(y=overall_mean, color='red', linestyle='--', alpha=0.7, linewidth=1.5, 
               label=f'Mean Recall ({overall_mean:.3f})')
    
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(axis='y', alpha=0.3, linestyle='-', linewidth=0.5)
    
    # Highlight N1 (often the hardest class)
    n1_idx = CLASS_ORDER.index('N1')
    bars[n1_idx].set_edgecolor('red')
    bars[n1_idx].set_linewidth(2)
    
    plt.tight_layout()
    
    # Save figure
    fig_path = OUTPUT_DIR / 'per_class_recall.png'
    plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"✓ Saved: {fig_path}")
    
    plt.show()
    
    # Print summary
    print(f"\n📊 Per-Class Recall Summary (Mean ± Std across {len(all_seed_results)} seeds):")
    for i, stage in enumerate(CLASS_ORDER):
        print(f"   {stage}: {mean_recalls[i]:.4f} ± {std_recalls[i]:.4f}")
    print(f"   Overall Mean: {overall_mean:.4f}")
    
else:
    print("❌ No results available. Run training first.")

## Section 9: Scientific Justification (For Reviewers)

### 9.1 Why Cross-Dataset Evaluation?

Cross-dataset evaluation tests **domain generalization** — whether features learned on one dataset transfer to a completely different dataset.

| Evaluation Type | Question Answered |
|-----------------|-------------------|
| **In-domain CV** | Subject generalization within the same dataset |
| **Cross-cohort** | Generalization to a different cohort (same dataset) |
| **Cross-dataset (this)** | Generalization to a completely different dataset |

---

### 9.2 Why This is a DOMAIN STRESS TEST

**This is NOT a performance estimation experiment.**

The goal is to measure:
- How much performance degrades under domain shift
- Which sleep stages are most robust to domain shift
- Lower bound of generalization capability

**Expected behavior:**
- Performance will be **significantly lower** than in-domain
- N1 detection typically suffers most (dataset-specific)
- Wake and N3 tend to be more robust

---

### 9.3 Domain Shift Factors

| Factor | Impact |
|--------|--------|
| **Population** | SHHS has older patients with cardiovascular risk |
| **Equipment** | Different amplifiers, sensors, sampling rates |
| **Protocol** | Different annotation guidelines and scorers |
| **Channels** | Different EEG montage (Fpz-Cz vs C3-A2) |

---

### 9.4 Why Multi-Seed Evaluation?

Random initialization affects:
- XGBoost tree construction
- SVC optimization path
- Ensemble calibration

Using 5 seeds provides:
- Robust mean estimate
- Variance quantification (±std)
- Confidence in reported numbers

---

### 9.5 Limitations

1. **Channel mismatch**: Sleep-EDF uses Fpz-Cz; SHHS uses C3-A2/C4-A1. Features may not align perfectly.

2. **Population bias**: Training on healthy subjects, testing on cardiovascular patients.

3. **No domain adaptation**: This is a zero-shot transfer — no fine-tuning on SHHS.

4. **Feature pipeline**: SHHS must use identical preprocessing to be valid.

## Section 10: Summary and Reproducibility

In [ ]:
# ===================================================================
# SUMMARY AND REPRODUCIBILITY STATEMENT
# ===================================================================

print("="*80)
print("CROSS-DATASET ABLATION SUMMARY")
print("="*80)

print(f"\n📋 EXPERIMENT CONFIGURATION")
print(f"─"*60)
print(f"EXPERIMENT_BOX:     {EXPERIMENT_BOX}")
print(f"DATASET_NAME:       {DATASET_NAME}")
print(f"RANDOM_SEEDS:       {RANDOM_SEEDS}")

print(f"\n📊 DATA")
print(f"─"*60)
print(f"Training dataset:   Sleep-EDF Expanded")
print(f"Training subjects:  {len(TRAIN_SUBJECT_IDS) if TRAIN_DATA_LOADED else 'N/A'}")
print(f"Test dataset:       SHHS")
print(f"Test subjects:      {len(TEST_SUBJECT_IDS) if TEST_DATA_LOADED else 'N/A'}")

if AGGREGATION_COMPLETE and aggregated is not None:
    print(f"\n📈 RESULTS (mean ± std across {aggregated['n_seeds']} seeds)")
    print(f"─"*60)
    print(f"Macro F1:           {aggregated['macro_f1_mean']:.4f} ± {aggregated['macro_f1_std']:.4f}")
    print(f"Balanced Accuracy:  {aggregated['balanced_accuracy_mean']:.4f} ± {aggregated['balanced_accuracy_std']:.4f}")
    print(f"Cohen's Kappa:      {aggregated['kappa_mean']:.4f} ± {aggregated['kappa_std']:.4f}")

print(f"\n" + "="*80)
print("REPRODUCIBILITY STATEMENT")
print("="*80)
print(f"""
EXPERIMENTAL PROTOCOL (CROSS-DATASET ABLATION)
───────────────────────────────────────────────
• EXPERIMENT_BOX:       {EXPERIMENT_BOX}
• DATASET_NAME:         {DATASET_NAME}
• RANDOM_SEEDS:         {RANDOM_SEEDS}

TRAINING
────────
• Training data:        Sleep-EDF Expanded (all subjects)
• Model:                XGBoost + LinearSVC ensemble (0.7/0.3)
• Scaler:               Fit on training data ONLY

TESTING
───────
• Test data:            SHHS (no training or tuning)
• Cross-validation:     NO (zero-shot transfer)
• Hyperparameter tuning: NO (frozen from Sleep-EDF training)

⚠️ CRITICAL SCOPE
─────────────────
This is a DOMAIN STRESS TEST, NOT a performance estimation experiment.
Performance degradation is EXPECTED and INFORMATIVE.
Results quantify domain shift, not optimal SHHS performance.
""")

if RESULTS_SAVED:
    print(f"\n📁 Results saved to: {OUTPUT_DIR}")
else:
    print(f"\n⚠️ Results not saved (data not available)")

print("="*80)